# 🛰️ SkyTrace: GPU-Accelerated Training & Evaluation Pipeline

This notebook runs the end-to-end building footprint segmentation fine-tuning pipeline on an NVIDIA GPU (e.g. Google Colab T4 / V100 / A100 or Kaggle GPU).

**Pipeline Steps:**
1. Verify GPU acceleration (`nvidia-smi`)
2. Clone & setup SkyTrace environment
3. Fetch SpaceNet-2 Paris satellite tiles & pretrained checkpoints
4. Fine-tune DeepLabV3+ with ResNet-50 backbone (Mixed Precision FP16)
5. Evaluate on untouched holdout test partition with zero data leakage
6. Export the fine-tuned model checkpoint & metrics

### Step 1: Check NVIDIA GPU Runtime

In [ ]:
!nvidia-smi

### Step 2: Clone SkyTrace Repository & Install Dependencies

In [ ]:
# Clone the latest code from GitHub
!git clone https://github.com/bathulasairamya/SkyTrace.git
%cd SkyTrace
!git pull origin main

# Install dependencies
!pip install -r requirements.txt

### Step 3: Download Pretrained Weights & SpaceNet-2 Dataset

In [ ]:
# Downloads best_deeplab.pth and acquires audited SpaceNet-2 Paris tiles
!python src/download_resources.py
!python src/split_data.py

### Step 4: Run GPU-Accelerated Fine-Tuning

* Batch size: `4`
* Epochs: `10` (Takes ~2 to 3 minutes on NVIDIA GPU with Mixed Precision)
* Loss: BCE + Soft Dice Loss
* Saves best checkpoint to `checkpoints/best_finetuned_model.pth`

In [ ]:
!python src/train.py --device cuda --epochs 10 --batch_size 4 --lr 0.0001

### Step 5: Evaluate & Benchmark on Untouched Holdout Test Partition

In [ ]:
!python src/evaluate.py --compare --device cuda

### Step 6: Display Training Curves & Evaluation Results

In [ ]:
from IPython.display import Image, display, Markdown

# Display loss & accuracy curves
if os.path.exists('logs/loss_curve.png'):
    display(Image('logs/loss_curve.png'))

# Display evaluation report
if os.path.exists('outputs/evaluation_results.md'):
    with open('outputs/evaluation_results.md') as f:
        display(Markdown(f.read()))

### Step 7: Download the Trained Model Checkpoint

In [ ]:
try:
    from google.colab import files
    files.download('checkpoints/best_finetuned_model.pth')
    files.download('logs/training_log.json')
    files.download('logs/loss_curve.png')
    files.download('outputs/evaluation_results.md')
    print('Download triggered successfully.')
except ImportError:
    print('Not in Google Colab environment. Checkpoint saved locally at checkpoints/best_finetuned_model.pth')